# Maps - Combining all countries into one
30/06/2026, Kuba Kowalski 

This notebook first forecefully integrates the Cote d'Ivoire and Rwanda (2012) provinces into the file of other 21 countries. Afterwards, it visualizes the combined datasets with the variants of birthplace and residence maps. The birthplace maps do not include Ethiopia due to the lack of birthplace variable in the selected sample years. 

In [1]:
%pip install scipy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
# packages
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from pathlib import Path
from scipy.stats import gaussian_kde
import pandas as pd


## Alternative approach to integrating Cote d'Ivoire and Rwanda 2012 in combined countries file.
Rwanda's 2012 sample uses different admin boundaries than the world file. The geom column appears to be differently encoded and PostGIS will reject and attempts at union between the world and rwanda files due to an apparent column mismatch despite the same column name and type. Just like Cote, it must added using an alternative approach. 

Cote was created by manually aggregating lvl2 units into provinces from a separate shapefile than the world one provided by IPUMS. This process introduced an error in the ID column of the cote file that refuses to be fixed in postgis and prevents it from being appended to the file combining other countries.

Solution is to just manually realign only the relevant columns and brute force the combine. It works with Python, but not PostGIS.

In [2]:
# Birthplace: add Côte d'Ivoire and Rwanda 2012

main = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all.geojson"
)

cote = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\04cote\geom_edu_birthplace_4cote.geojson"
)

rwanda = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\12rwanda\geom_edu_birthplace_12rwanda2012.geojson"
)

# minimal harmonisation
cote = cote.copy()
cote["country_id"] = 4
cote["country"] = "cote_divoire"

rwanda = rwanda.copy()
rwanda["country_id"] = 12
rwanda["country"] = "rwanda"

rename_map = {
    "nc_cote": "n",
    "nc_rwanda": "n",
}

cote = cote.rename(columns={k: v for k, v in rename_map.items() if k in cote.columns})
rwanda = rwanda.rename(columns={k: v for k, v in rename_map.items() if k in rwanda.columns})

# match CRS
if cote.crs != main.crs:
    cote = cote.to_crs(main.crs)

if rwanda.crs != main.crs:
    rwanda = rwanda.to_crs(main.crs)

# Rwanda 2012 geometry uses PROV2012 instead of GEOLEVEL1
if "GEOLEVEL1" not in rwanda.columns and "PROV2012" in rwanda.columns:
    rwanda = rwanda.rename(columns={"PROV2012": "GEOLEVEL1"})

rwanda["GEOLEVEL1"] = rwanda["GEOLEVEL1"].astype(str).str.strip().str.zfill(6)

# keep only columns that already exist in the main file
common_cols = [c for c in main.columns if c in cote.columns and c in rwanda.columns]

combined = gpd.GeoDataFrame(
    pd.concat(
        [main[common_cols], cote[common_cols], rwanda[common_cols]],
        ignore_index=True
    ),
    geometry="geometry",
    crs=main.crs
)

combined.to_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson",
    driver="GeoJSON"
)

print(len(main), "rows in original birthplace")
print(len(cote), "rows in cote birthplace")
print(len(rwanda), "rows in rwanda birthplace")
print(len(combined), "rows in combined birthplace")
print(combined["country"].unique())

1947 rows in original birthplace
63 rows in cote birthplace
35 rows in rwanda birthplace
2045 rows in combined birthplace
['benin' 'botswana' 'burkina_faso' 'ghana' 'guinea' 'kenya' 'malawi'
 'mali' 'mozambique' 'senegal' 'sierra_leone' 'tanzania' 'togo' 'uganda'
 'zambia' 'liberia' 'cameroon' 'south_sudan' 'sudan' 'zimbabwe'
 'cote_divoire' 'rwanda']


In [3]:
# Residence: add Côte d'Ivoire and Rwanda 2012

main = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all.geojson"
)

cote = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\04cote\geom_edu_residence_4cote.geojson"
)

rwanda = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\12rwanda\geom_edu_residence_12rwanda2012.geojson"
)

cote = cote.copy()
cote["country_id"] = 4
cote["country"] = "cote_divoire"

rwanda = rwanda.copy()
rwanda["country_id"] = 12
rwanda["country"] = "rwanda"

rename_map = {
    "nc_cote": "n",
    "nc_rwanda": "n",
}

cote = cote.rename(columns={k: v for k, v in rename_map.items() if k in cote.columns})
rwanda = rwanda.rename(columns={k: v for k, v in rename_map.items() if k in rwanda.columns})

if cote.crs != main.crs:
    cote = cote.to_crs(main.crs)

if rwanda.crs != main.crs:
    rwanda = rwanda.to_crs(main.crs)

# Rwanda 2012 geometry uses PROV2012 instead of GEOLEVEL1
if "GEOLEVEL1" not in rwanda.columns and "PROV2012" in rwanda.columns:
    rwanda = rwanda.rename(columns={"PROV2012": "GEOLEVEL1"})

rwanda["GEOLEVEL1"] = rwanda["GEOLEVEL1"].astype(str).str.strip().str.zfill(6)    

common_cols = [c for c in main.columns if c in cote.columns and c in rwanda.columns]

combined = gpd.GeoDataFrame(
    pd.concat(
        [main[common_cols], cote[common_cols], rwanda[common_cols]],
        ignore_index=True
    ),
    geometry="geometry",
    crs=main.crs
)

combined.to_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson",
    driver="GeoJSON"
)

print(len(main), "rows in original residence")
print(len(cote), "rows in cote residence")
print(len(rwanda), "rows in rwanda residence")
print(len(combined), "rows in combined residence")
print(combined["country"].unique())

2027 rows in original residence
63 rows in cote residence
35 rows in rwanda residence
2125 rows in combined residence
['benin' 'botswana' 'burkina_faso' 'ethiopia' 'ghana' 'guinea' 'kenya'
 'malawi' 'mali' 'mozambique' 'senegal' 'sierra_leone' 'tanzania' 'togo'
 'uganda' 'zambia' 'liberia' 'cameroon' 'south_sudan' 'sudan' 'zimbabwe'
 'cote_divoire' 'rwanda']


## Check
If Cote and Rwanda not in output list, you messed up. 

In [4]:
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_higher_maps"
)
output_dir.mkdir(exist_ok=True)

gdf = gpd.read_file(input_file)
africa_outline = gpd.read_file(africa_outline_file)

print(len(gdf), "rows loaded from combined residence file")
print(gdf["country"].unique())

2125 rows loaded from combined residence file
['benin' 'botswana' 'burkina_faso' 'ethiopia' 'ghana' 'guinea' 'kenya'
 'malawi' 'mali' 'mozambique' 'senegal' 'sierra_leone' 'tanzania' 'togo'
 'uganda' 'zambia' 'liberia' 'cameroon' 'south_sudan' 'sudan' 'zimbabwe'
 'cote_divoire' 'rwanda']


## Birthplace/primary - series per cohort

In [5]:
# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_primary_maps"
)
output_dir.mkdir(exist_ok=True)

# ---- load data ----
gdf = gpd.read_file(input_file)
africa = gpd.read_file(africa_outline_file)

# ---- CRS alignment ----
if africa.crs != gdf.crs:
    africa = africa.to_crs(gdf.crs)

# ---- clean data ----
gdf["cohort"] = gdf["cohort"].astype("Int64")
gdf["primary_educ"] = gdf["primary_educ"].astype(float)

# ---- keep valid rows ----
gdf = gdf[gdf["cohort"].notna() & gdf["primary_educ"].notna()].copy()

# ---- combined scale across all countries including Côte d'Ivoire ----
vmin = gdf["primary_educ"].min()
vmax = gdf["primary_educ"].max()

# ---- cohorts present in combined dataset ----
all_cohorts = sorted(gdf["cohort"].dropna().unique())

for cohort in all_cohorts:
    subset = gdf[gdf["cohort"] == cohort].copy()
    values = subset["primary_educ"].dropna().values

    if len(values) == 0:
        continue

    fig, ax = plt.subplots(figsize=(16, 20))

    # ---- Africa background outline ----
    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    # ---- combined layer, now including Côte d'Ivoire ----
    subset.plot(
        column="primary_educ",
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(f"Birthplace-based primary education, cohort {cohort}")
    ax.set_axis_off()

    # fixed Africa-wide extent
    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    # ---- horizontal colorbar ----
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )
    cbar.set_label("Primary education share")

    # ---- wave axis above colorbar ----
    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5, 0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = output_dir / f"birthplace_primary_educ_{cohort}.png"
    plt.savefig(out_file, dpi=800, bbox_inches="tight")
    plt.close()

print("Done. Maps saved to:", output_dir)

Done. Maps saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_primary_maps


## Birthplace/higher - series per cohort

In [6]:
# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_higher_maps")
output_dir.mkdir(exist_ok=True)

# ---- load data ----
gdf = gpd.read_file(input_file)
africa = gpd.read_file(africa_outline_file)

# ---- CRS alignment ----
if africa.crs != gdf.crs:
    africa = africa.to_crs(gdf.crs)

# ---- clean data ----
gdf["cohort"] = gdf["cohort"].astype("Int64")
gdf["higher_educ"] = gdf["higher_educ"].astype(float)

# ---- keep valid rows ----
gdf = gdf[gdf["cohort"].notna() & gdf["higher_educ"].notna()].copy()

# ---- scale across all countries, including Côte d'Ivoire ----
vmin = gdf["higher_educ"].min()
vmax = gdf["higher_educ"].max()

# ---- cohorts present in combined dataset ----
all_cohorts = sorted(gdf["cohort"].dropna().unique())

for cohort in all_cohorts:
    subset = gdf[gdf["cohort"] == cohort].copy()
    values = subset["higher_educ"].dropna().values

    if len(values) == 0:
        continue

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    subset.plot(
        column="higher_educ",
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(f"Birthplace-based higher education, cohort {cohort}")
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )
    cbar.set_label("Higher education share")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5, 0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = output_dir / f"birthplace_higher_educ_{cohort}.png"
    plt.savefig(out_file, dpi=800, bbox_inches="tight")
    plt.close()

print("Done. Maps saved to:", output_dir)

Done. Maps saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_higher_maps


## Birthplace/tertiary - series per cohort

In [7]:
# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_tertiary_maps")
output_dir.mkdir(exist_ok=True)

# ---- load data ----
gdf = gpd.read_file(input_file)
africa = gpd.read_file(africa_outline_file)

# ---- CRS alignment ----
if africa.crs != gdf.crs:
    africa = africa.to_crs(gdf.crs)

# ---- clean data ----
gdf["cohort"] = gdf["cohort"].astype("Int64")
gdf["tertiary_educ"] = gdf["tertiary_educ"].astype(float)

# ---- keep valid rows ----
gdf = gdf[gdf["cohort"].notna() & gdf["tertiary_educ"].notna()].copy()

# ---- scale across all countries, including Côte d'Ivoire ----
vmin = gdf["tertiary_educ"].min()
vmax = gdf["tertiary_educ"].max()

# ---- cohorts present in combined dataset ----
all_cohorts = sorted(gdf["cohort"].dropna().unique())

for cohort in all_cohorts:
    subset = gdf[gdf["cohort"] == cohort].copy()
    values = subset["tertiary_educ"].dropna().values

    if len(values) == 0:
        continue

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    subset.plot(
        column="tertiary_educ",
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(f"Birthplace-based tertiary education, cohort {cohort}")
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )
    cbar.set_label("Tertiary education share")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5, 0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = output_dir / f"birthplace_tertiary_educ_{cohort}.png"
    plt.savefig(out_file, dpi=800, bbox_inches="tight")
    plt.close()

print("Done. Maps saved to:", output_dir)

Done. Maps saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_tertiary_maps


## Residence/primary - series per cohort

In [8]:
# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_primary_maps")
output_dir.mkdir(exist_ok=True)

# ---- load data ----
gdf = gpd.read_file(input_file)
africa = gpd.read_file(africa_outline_file)

# ---- CRS alignment ----
if africa.crs != gdf.crs:
    africa = africa.to_crs(gdf.crs)

# ---- clean data ----
gdf["cohort"] = gdf["cohort"].astype("Int64")
gdf["primary_educ"] = gdf["primary_educ"].astype(float)

# ---- keep valid rows ----
gdf = gdf[gdf["cohort"].notna() & gdf["primary_educ"].notna()].copy()

# ---- combined scale across all countries including Côte d'Ivoire ----
vmin = gdf["primary_educ"].min()
vmax = gdf["primary_educ"].max()

# ---- cohorts present in combined dataset ----
all_cohorts = sorted(gdf["cohort"].dropna().unique())

for cohort in all_cohorts:
    subset = gdf[gdf["cohort"] == cohort].copy()
    values = subset["primary_educ"].dropna().values

    if len(values) == 0:
        continue

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    subset.plot(
        column="primary_educ",
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(f"Residence-based primary education, cohort {cohort}")
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )
    cbar.set_label("Primary education share")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5, 0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = output_dir / f"residence_primary_educ_{cohort}.png"
    plt.savefig(out_file, dpi=800, bbox_inches="tight")
    plt.close()

print("Done. Maps saved to:", output_dir)

Done. Maps saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_primary_maps


## residence/higher

In [9]:
# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_higher_maps")
output_dir.mkdir(exist_ok=True)

# ---- load data ----
gdf = gpd.read_file(input_file)
africa = gpd.read_file(africa_outline_file)

# ---- CRS alignment ----
if africa.crs != gdf.crs:
    africa = africa.to_crs(gdf.crs)

# ---- clean data ----
gdf["cohort"] = gdf["cohort"].astype("Int64")
gdf["higher_educ"] = gdf["higher_educ"].astype(float)

# ---- keep valid rows ----
gdf = gdf[gdf["cohort"].notna() & gdf["higher_educ"].notna()].copy()

# ---- combined scale across all countries including Côte d'Ivoire ----
vmin = gdf["higher_educ"].min()
vmax = gdf["higher_educ"].max()

# ---- cohorts present in combined dataset ----
all_cohorts = sorted(gdf["cohort"].dropna().unique())

for cohort in all_cohorts:
    subset = gdf[gdf["cohort"] == cohort].copy()
    values = subset["higher_educ"].dropna().values

    if len(values) == 0:
        continue

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    subset.plot(
        column="higher_educ",
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(f"Residence-based higher education, cohort {cohort}")
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )
    cbar.set_label("Higher education share")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5, 0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = output_dir / f"residence_higher_educ_{cohort}.png"
    plt.savefig(out_file, dpi=800, bbox_inches="tight")
    plt.close()

print("Done. Maps saved to:", output_dir)

Done. Maps saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_higher_maps


## Residence/tertiary

In [10]:
# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_tertiary_maps")
output_dir.mkdir(exist_ok=True)

# ---- load data ----
gdf = gpd.read_file(input_file)
africa = gpd.read_file(africa_outline_file)

# ---- CRS alignment ----
if africa.crs != gdf.crs:
    africa = africa.to_crs(gdf.crs)

# ---- clean data ----
gdf["cohort"] = gdf["cohort"].astype("Int64")
gdf["tertiary_educ"] = gdf["tertiary_educ"].astype(float)

# ---- keep valid rows ----
gdf = gdf[gdf["cohort"].notna() & gdf["tertiary_educ"].notna()].copy()

# ---- combined scale across all countries including Côte d'Ivoire ----
vmin = gdf["tertiary_educ"].min()
vmax = gdf["tertiary_educ"].max()

# ---- cohorts present in combined dataset ----
all_cohorts = sorted(gdf["cohort"].dropna().unique())

for cohort in all_cohorts:
    subset = gdf[gdf["cohort"] == cohort].copy()
    values = subset["tertiary_educ"].dropna().values

    if len(values) == 0:
        continue

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    subset.plot(
        column="tertiary_educ",
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(f"Residence-based tertiary education, cohort {cohort}")
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )
    cbar.set_label("Tertiary education share")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5, 0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = output_dir / f"residence_tertiary_educ_{cohort}.png"
    plt.savefig(out_file, dpi=800, bbox_inches="tight")
    plt.close()

print("Done. Maps saved to:", output_dir)

Done. Maps saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_tertiary_maps


## Birthplace - table export 

In [11]:
# ---- Export combined birthplace dataset (without geometry) to Excel ----

import geopandas as gpd
from pathlib import Path

# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson"

output_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\edu_birthplace_all_no_geom.xlsx"
)

# ---- load combined dataset ----
gdf = gpd.read_file(input_file)

# ---- remove geometry ----
df = gdf.drop(columns="geometry", errors="ignore")

# ---- export ----
df.to_excel(output_file, index=False)

print(f"Exported {len(df):,} rows")
print("Excel file saved to:", output_file)

Exported 2,045 rows
Excel file saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\edu_birthplace_all_no_geom.xlsx


## Table exports (without the spatial component)

In [12]:
# ---- Export combined residence dataset (without geometry) to Excel ----

import geopandas as gpd
from pathlib import Path

input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"

output_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\edu_residence_all_no_geom.xlsx"
)

gdf = gpd.read_file(input_file)

df = gdf.drop(columns="geometry", errors="ignore")

df.to_excel(output_file, index=False)

print(f"Exported {len(df):,} rows")
print("Excel file saved to:", output_file)

Exported 2,125 rows
Excel file saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\edu_residence_all_no_geom.xlsx


In [13]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ---- paths ----
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all.geojson"
cote_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\04cote\geom_edu_birthplace_4cote.geojson"

output_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_no_geom.xlsx"
)

# ---- load ----
gdf_all = gpd.read_file(input_file)
cote = gpd.read_file(cote_file)

# ---- Côte d'Ivoire identifiers ----
cote["country_id"] = 4
cote["country"] = "cote_divoire"

# ---- remove geometry ----
gdf_all = gdf_all.drop(columns="geometry", errors="ignore")
cote = cote.drop(columns="geometry", errors="ignore")

# ---- combine ----
all_columns = sorted(set(gdf_all.columns) | set(cote.columns))
gdf_all = gdf_all.reindex(columns=all_columns)
cote = cote.reindex(columns=all_columns)

df = pd.concat([gdf_all, cote], ignore_index=True)

# ---- merge nc_cote into n ----
if "nc_cote" in df.columns:
    if "n" not in df.columns:
        df["n"] = pd.NA

    df["n"] = df["n"].combine_first(df["nc_cote"])
    df = df.drop(columns=["nc_cote"])

# ---- remove gid ----
df = df.drop(columns=["gid"], errors="ignore")

# ---- fix uid for Côte d'Ivoire / missing uid rows ----
if "uid" not in df.columns:
    df["uid"] = pd.NA

df["uid"] = pd.to_numeric(df["uid"], errors="coerce").astype("Int64")

missing_uid = df["uid"].isna()
df.loc[missing_uid, "uid"] = range(1678, 1678 + missing_uid.sum())

# ---- optional: put key columns first ----
first_cols = [c for c in ["uid", "country_id", "country", "GEOLEVEL1", "cohort", "primary_educ", "higher_educ", "tertiary_educ", "n"] if c in df.columns]
other_cols = [c for c in df.columns if c not in first_cols]
df = df[first_cols + other_cols]

# ---- export ----
df.to_excel(output_file, index=False)

print("Excel file saved to:", output_file)

Excel file saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_no_geom.xlsx
